# 01 — Exploratory Data Analysis
## College Complaint Intelligence System

> **Status:** Phase 2 | Completed  
> **Dataset:** `data/raw/college_complaints_synthetic.csv` (`data_source=synthetic`)  
> **Purpose:** Inspect class distributions, text lengths, missing values, duplicate incidents, and vocabulary patterns.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.load_data import load_config, load_csv
from src.data.validate_data import validate_dataframe

# Set display and plotting styles
pd.set_option('display.max_colwidth', None)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

### 1. Load Dataset & Configuration

In [ ]:
cfg = load_config("config.yaml")
df = load_csv("data/raw/college_complaints_synthetic.csv")
print(f"Total records loaded: {len(df)}")
df.head(3)

### 2. Validation Check

In [ ]:
report = validate_dataframe(
    df,
    text_column=cfg["data"]["text_column"],
    label_column=cfg["data"]["label_column"],
    valid_labels=list(cfg["categories"].keys()),
    min_text_length=cfg["data"]["min_text_length"],
    min_class_examples=cfg["category_min_examples"],
    pii_patterns=cfg["preprocessing"]["pii_patterns"],
)
print(report.summary())

### 3. Category Distribution

In [ ]:
cat_counts = df["category"].value_counts().sort_index()
plt.figure(figsize=(10, 4.5))
sns.barplot(x=cat_counts.index, y=cat_counts.values, color="#2b5c8f")
plt.title("Complaint Category Distribution")
plt.ylabel("Count")
plt.xlabel("Category Code")
plt.show()

### 4. Text Length Analysis

In [ ]:
df["word_cnt"] = df["complaint_text"].str.split().str.len()
print(df["word_cnt"].describe())
plt.figure(figsize=(10, 4))
sns.histplot(df["word_cnt"], bins=25, kde=True, color="#319795")
plt.title("Distribution of Complaint Word Counts")
plt.xlabel("Words per Complaint")
plt.show()

### 5. Multi-Report Incident Groups (Duplicates / Related Incidents)

In [ ]:
grouped = df[df["incident_group_id"].fillna("").str.strip() != ""]
print(f"Grouped Incident Reports: {len(grouped)} records across {grouped['incident_group_id'].nunique()} incidents")
grouped[["incident_group_id", "complaint_text", "category", "created_at"]].head(6)